In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
from time import time
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
print("Libraries imported. Running full analysis to generate results...")

# --- 1. Load Data ---
df = pd.read_csv("../data/final_model_dataset_v3.csv")

# --- 2. Define Features (X) and Target (y) ---
ignore_cols = ["cation", "cation_ion_name", "anion", "anion_ion_name"]
target_col = "CO2_solubility"
feature_cols = [col for col in df.columns if col not in ignore_cols + [target_col]]
X = df[feature_cols]
y = df[target_col].values

# --- 3. Split and Scale Data ---
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# --- 4. Define All Four Models ---
def create_model(meta):
    n_features_in_ = meta["n_features_in_"]
    model = keras.Sequential([
        layers.Input(shape=(n_features_in_,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Neural Network": KerasRegressor(model=create_model, epochs=100, batch_size=32, verbose=0)
}

# --- 5. Train and Evaluate All Models ---
results = {}
print("Training and evaluating all models...")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    if hasattr(y_pred, 'flatten'): y_pred = y_pred.flatten()
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = root_mean_squared_error(y_val, y_pred)
    results[name] = {"R²": r2, "MAE": mae, "RMSE": rmse}
results_df = pd.DataFrame(results).T

# --- 6. Calculate Feature Importance ---
print("Calculating feature importance for all models...")
importance_results = {}
for name, model in models.items():
    perm_importance = permutation_importance(
        model, X_val_scaled, y_val, 
        n_repeats=10, random_state=42, n_jobs=-1
    )
    importance_results[name] = perm_importance.importances_mean
importance_df = pd.DataFrame(importance_results, index=feature_cols)

print("\n--- Analysis Complete. Ready to generate tables. ---")

Libraries imported. Running full analysis to generate results...
Training and evaluating all models...
Calculating feature importance for all models...

--- Analysis Complete. Ready to generate tables. ---


In [2]:
# --- Table 1: Performance Metrics ---
print("--- Table 1: Performance Metrics of Four Machine Learning Algorithms on Independent Hold-Out Test Set ---")

# Rename columns to match your request
table1_df = results_df[['R²', 'MAE', 'RMSE']].copy()
table1_df.columns = ['R²', 'MAE (mol CO₂/mol IL)', 'RMSE (mol CO₂/mol IL)']
table1_df.index.name = 'Algorithm'
table1_df = table1_df.sort_values(by='R²', ascending=False)

# Print as markdown for a nice table format
print(table1_df.to_markdown(floatfmt=".4f"))

--- Table 1: Performance Metrics of Four Machine Learning Algorithms on Independent Hold-Out Test Set ---
| Algorithm         |     R² |   MAE (mol CO₂/mol IL) |   RMSE (mol CO₂/mol IL) |
|:------------------|-------:|-----------------------:|------------------------:|
| Random Forest     | 0.9853 |                 0.0167 |                  0.0275 |
| Gradient Boosting | 0.9729 |                 0.0257 |                  0.0374 |
| Neural Network    | 0.9687 |                 0.0329 |                  0.0402 |
| Ridge Regression  | 0.5906 |                 0.1164 |                  0.1455 |


In [4]:
# --- Tables 2-5: Feature Importance Rankings (Excluding Atom Counts) ---

table_number = 2
for model_name in models.keys(): # Loop through 'Ridge Regression', 'Random Forest', etc.
    print(f"\n--- Table {table_number}: Feature Importance Rankings for {model_name} (Excluding Atom Counts) ---")
    
    # Get the full importance list for the model
    all_ranked_features = importance_df[model_name].sort_values(ascending=False)
    
    # --- NEW: Filter the list to exclude all atom count columns ---
    is_atom_count = all_ranked_features.index.str.startswith('anion_num_') | \
                    all_ranked_features.index.str.startswith('cation_num_')
    # Select only the features that are NOT atom counts
    ranked_features = all_ranked_features[~is_atom_count]
    # --- End of modification ---
    
    # Create a DataFrame
    table_df = pd.DataFrame(ranked_features)
    table_df.reset_index(inplace=True)
    table_df.columns = ['Feature', 'Importance Score']
    
    # Add the Rank column (starting from 1)
    table_df['Rank'] = table_df.index + 1
    
    # Reorder columns to match your request
    table_df = table_df[['Rank', 'Feature', 'Importance Score']]
    
    # Print the Top 15 features in markdown format
    print(table_df.head(15).to_markdown(index=False, floatfmt=".6f"))
    
    table_number += 1


--- Table 2: Feature Importance Rankings for Ridge Regression (Excluding Atom Counts) ---
|   Rank | Feature                    |   Importance Score |
|-------:|:---------------------------|-------------------:|
|      1 | Pressure_kPa               |           0.938275 |
|      2 | Temperature_K              |           0.045139 |
|      3 | anion_Energy_Hartree       |           0.012743 |
|      4 | cation_LUMO_Eh             |           0.010169 |
|      5 | anion_LUMO_Eh              |           0.006910 |
|      6 | anion_Dipole_Moment_Debye  |           0.003860 |
|      7 | anion_HOMO_LUMO_Gap_Eh     |           0.003540 |
|      8 | cation_HOMO_Eh             |           0.002773 |
|      9 | anion_HOMO_Eh              |           0.001660 |
|     10 | cation_HOMO_LUMO_Gap_Eh    |           0.001549 |
|     11 | cation_Energy_Hartree      |           0.000288 |
|     12 | cation_Dipole_Moment_Debye |          -0.000338 |

--- Table 3: Feature Importance Rankings for Random Fo

In [5]:
import pandas as pd

# Load the final dataset
df = pd.read_csv("../data/final_model_dataset_v3.csv")

# Create a list of unique 'cation-anion' pairs
unique_ils = df.drop_duplicates(subset=['cation', 'anion'])

print(f"The dataset contains {len(df)} total experimental data points.")
print(f"These data points cover {len(unique_ils)} unique ionic liquids.")

The dataset contains 11256 total experimental data points.
These data points cover 113 unique ionic liquids.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.